# Bottleneck Adapters
Prerequisites: matrix multiplication and residual layers. Estimated time: 35 minutes. Objectives: implement a bottleneck adapter, compare its trainable parameter count with a dense transform, and diagnose edge cases.

## Concept recap and assumptions
A bottleneck adapter keeps the input representation, projects it to a smaller dimension, applies a nonlinearity, projects back, and adds the result residually. The frozen backbone is represented here only by its incoming vector; this simulation does not train a neural network or measure model quality.

In [ ]:
def matvec(matrix, vector):
    if not matrix or any(len(row) != len(vector) for row in matrix):
        raise ValueError('matrix and vector dimensions must agree')
    return [sum(w * x for w, x in zip(row, vector)) for row in matrix]

def bottleneck_adapter(x, down, up, bias_down=None, bias_up=None):
    if not down or not up or len(down[0]) != len(x) or len(up) != len(x) or any(len(row) != len(down) for row in up):
        raise ValueError('adapter dimensions do not match input')
    bias_down = [0.0] * len(down) if bias_down is None else bias_down
    bias_up = [0.0] * len(up) if bias_up is None else bias_up
    if len(bias_down) != len(down) or len(bias_up) != len(up):
        raise ValueError('bias dimensions do not match projections')
    hidden = [max(0.0, v + b) for v, b in zip(matvec(down, x), bias_down)]
    delta = [v + b for v, b in zip(matvec(up, hidden), bias_up)]
    return [a + b for a, b in zip(x, delta)]

x = [1.0, -2.0]
down = [[1.0, 0.0]]
up = [[0.5], [-1.0]]
adapted = bottleneck_adapter(x, down, up)
adapted

## Baseline behavior
The residual path preserves the original representation while the narrow branch learns a task-specific update. A zero up-projection is a common initialization pattern: at initialization, the adapter is an identity mapping.

In [ ]:
assert adapted == [1.5, -3.0]
zero_up = [[0.0], [0.0]]
assert bottleneck_adapter(x, down, zero_up) == x
# For d input/output and bottleneck width r, two projections have 2*d*r weights.
d, r = 768, 32
adapter_weights = 2 * d * r
dense_weights = d * d
assert adapter_weights == 49152 and adapter_weights < dense_weights

## Comparison and metric
This parameter-count comparison excludes biases, layer norms, the frozen backbone, optimizer state, and activations. It estimates trainable projection weights only; it is not a memory or latency benchmark. Smaller bottlenecks save parameters but may constrain the update.

In [ ]:
def edge_case_checks():
    try:
        matvec([[1.0, 2.0]], [1.0])
    except ValueError:
        dimension_error = True
    else:
        dimension_error = False
    assert dimension_error  # malformed projection rejected
    try:
        bottleneck_adapter([2.0], [], [])
    except ValueError:
        empty_projection_error = True
    else:
        empty_projection_error = False
    assert empty_projection_error
    assert bottleneck_adapter([2.0], [[-1.0]], [[3.0]]) == [2.0]  # negative hidden pre-activation is clipped by ReLU
edge_case_checks()

## Exercises
1. Change the bottleneck width and derive the projection-weight count for `d=512`. 2. Add a nonzero down-projection bias and test one negative pre-activation. 3. In a real training setup, what must be frozen and what held-out comparison would detect an adapter that underfits? Expected properties: dimensions remain valid, parameter count scales linearly with bottleneck width, and the residual path remains present.

## Summary, next lesson, and practice
A bottleneck adapter is a narrow nonlinear residual branch, distinct from LoRA's low-rank weight update. This deterministic example checks mechanics only. Next: Prefix Tuning and Parameter-Efficient Fine-Tuning. Practice: [Adapters interview drill](../interview-prep/curriculum-practice.md#adapters).